In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("Housing.csv")
df.head(5)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [64]:
df['furnishingstatus'].value_counts()

furnishingstatus
semi-furnished    227
unfurnished       178
furnished         140
Name: count, dtype: int64

price-> Standard Scaler
area -> standard Scaler
mainroad -> OHE 
guestroom -> OHE
basement -> OHE
hotwaterheating->OHE
airconditioning -> OHE
prefarea -> OHE
furnishingstatus -> Ordinal Encoding

In [10]:
df.shape

(545, 13)

In [12]:
df.dtypes

price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object

In [16]:
df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [18]:
X = df.drop('price',axis = 1)
y = df['price']


In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)


In [68]:
numeric_features = ['area','bedrooms','bathrooms','stories','parking']
cat_feat = ["mainroad","guestroom","basement","hotwaterheating","airconditioning","prefarea"]
cat_ord = ["furnishingstatus"]

In [70]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [76]:
numerical_transformer = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = "median")),
        ('standard scaler', StandardScaler())
    ]
)

categorical_onehot_transformer = Pipeline(
    steps = [
        ("one hot encoding", OneHotEncoder(drop='first',sparse_output=False)) 
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps = [
        ("Ordinal encoder", OrdinalEncoder(categories= [['unfurnished','semi-furnished','furnished']]))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ("num transformer", numerical_transformer, numeric_features),
        ("categorical non diff", categorical_onehot_transformer, cat_feat),
        ("categorical diff", categorical_ordinal_transformer, cat_ord)
    ], remainder="passthrough"
)


In [78]:
X_train_transformed = preprocessor.fit_transform(X_train)

In [80]:
X_test_transformed = preprocessor.transform(X_test)

In [82]:
print(X_train_transformed.shape)

(436, 12)


In [84]:
print(X_test_transformed.shape)

(109, 12)


In [86]:
from sklearn.linear_model import LinearRegression

In [88]:
lr = Pipeline(
    steps = [
        ("preprocessing", preprocessor),
        ("model_training", LinearRegression())
    ]
)

In [90]:
lr.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num transformer',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standard '
                                                                   'scaler',
                                                                   StandardScaler())]),
                                                  ['area', 'bedrooms',
                                                   'bathrooms', 'stories',
                                                   'parking']),
                                                 ('categorical non diff',
                                                  Pipeline(steps=[('one hot '
                                                                   'encoding',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  ['mainroad', 'guestroom',
                                                   'basement',
                                                   'hotwaterheating',
                                                   'airconditioning',
                                                   'prefarea']),
                                                 ('categorical diff',
                                                  Pipeline(steps=[('Ordinal '
                                                                   'encoder',
                                                                   OrdinalEncoder(categories=[['unfurnished',
                                                                                               'semi-furnished',
                                                                                               'furnished']]))]),
                                                  ['furnishingstatus'])])),
                ('model_training', LinearRegression())])

In [94]:
lr.predict(X_test)

array([5203691.70963177, 7257004.02115475, 3062828.59668172,
       4559591.65374424, 3332932.30559783, 3563080.67918996,
       5645466.3121997 , 6413979.66873635, 2755831.54819001,
       2668938.6607523 , 9570600.29915351, 2827431.50860062,
       3195686.25834091, 3352263.99438472, 3713879.49996132,
       5301088.2443575 , 2987920.26669681, 4810799.8212371 ,
       4383031.70489929, 3525092.18938646, 5796259.50068013,
       5840000.702993  , 2760214.60864101, 4762590.14920608,
       5204755.73895205, 7515542.71619022, 3254681.68956383,
       5236164.45964445, 8178523.1682028 , 3434166.1567565 ,
       6443921.58767582, 3346004.77919185, 6742324.74004132,
       4154936.84088665, 3589152.47491253, 5788125.92515323,
       4768370.18154077, 4391684.04193173, 3217657.04549936,
       4638196.61928879, 4522160.27786713, 3541284.06127246,
       7238136.1194117 , 4021515.68926614, 3701978.76822756,
       4298879.55563098, 6705004.0206061 , 3993466.52296897,
       3798185.05328058,

In [100]:
from sklearn.metrics import mean_squared_error,r2_score,root_mean_squared_error,mean_absolute_error
y_pred = lr.predict(X_test)


In [102]:
print(f"MSE: {round(mean_squared_error(y_test, y_pred), 4)}")
print(f"R2: {round(r2_score(y_test, y_pred), 4)}")
print(f"RMSE: {round(root_mean_squared_error(y_test, y_pred), 4)}")
print(f"MAE: {round(mean_absolute_error(y_test, y_pred), 4)}")

MSE: 1771751116594.0398
R2: 0.6495
RMSE: 1331071.4168
MAE: 979679.6913


In [106]:
from sklearn.linear_model import SGDRegressor

In [108]:
SGD_pipe= Pipeline( steps=[ ('preprocessor',preprocessor), ('model' , SGDRegressor()) ])


In [112]:
SGD_pipe.fit(X_test,y_test)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num transformer',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standard '
                                                                   'scaler',
                                                                   StandardScaler())]),
                                                  ['area', 'bedrooms',
                                                   'bathrooms', 'stories',
                                                   'parking']),
                                                 ('categorical non diff',
                                                  Pipeline(steps=[('one hot '
                                                                   'encoding',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  ['mainroad', 'guestroom',
                                                   'basement',
                                                   'hotwaterheating',
                                                   'airconditioning',
                                                   'prefarea']),
                                                 ('categorical diff',
                                                  Pipeline(steps=[('Ordinal '
                                                                   'encoder',
                                                                   OrdinalEncoder(categories=[['unfurnished',
                                                                                               'semi-furnished',
                                                                                               'furnished']]))]),
                                                  ['furnishingstatus'])])),
                ('model', SGDRegressor())])

In [114]:
y_pred = SGD_pipe.predict(X_test)